In [ ]:
conda install tensorflow-gpu

In [ ]:
import torch

import os
import nibabel as nib
import numpy as np
from torch.utils.data import Dataset, DataLoader
import torch

In [ ]:
import os
import numpy as np
import nibabel as nib
import torch
from torch.utils.data import Dataset

class Custom2DBraTSDataset(Dataset):
    def __init__(self, data_dir, modality):
        self.data_dir = data_dir
        self.modality = modality
        self.patient_ids = [d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))]

        # Initialize lists to store slices
        self.images = []
        self.labels = []

        # Iterate through patients and load slices
        for patient_id in self.patient_ids:
            patient_path = os.path.join(self.data_dir, patient_id)

            # Load image and label volumes
            image = nib.load(os.path.join(patient_path, f'{patient_id}_{self.modality}.nii.gz')).get_fdata()
            label = nib.load(os.path.join(patient_path, f'{patient_id}_seg.nii.gz')).get_fdata()

            # Append all slices to the list
            for slice_idx in range(image.shape[2] // 2 - 20, image.shape[2] // 2 + 20):
                image_slice = image[:, :, slice_idx]
                label_slice = label[:, :, slice_idx]

                # Convert to torch tensor and add channel dimension for image
                image_tensor = torch.tensor(image_slice, dtype=torch.float32).unsqueeze(0)  # Add channel dimension
                # rgb_image_tensor = torch.cat((image_tensor, image_tensor, image_tensor), dim=0)
                label_tensor = torch.tensor(label_slice, dtype=torch.long)

                self.images.append(image_tensor)
                self.labels.append(label_tensor)

    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]
        return image, label

    def __len__(self):
        return len(self.images)

In [ ]:
# class CustomBraTSDataset(Dataset):
#     def __init__(self, data_dir, modality):
#         self.data_dir = data_dir
#         self.modality = modality
#         self.patient_ids = [d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))]

#     def __getitem__(self, idx):
#         patient_id = self.patient_ids[idx]
#         patient_path = os.path.join(self.data_dir, patient_id)

#         image = nib.load(os.path.join(patient_path, f'{patient_id}_{self.modality}.nii.gz')).get_fdata()

#         # Load the ground truth segmentation
#         label = nib.load(os.path.join(patient_path, f'{patient_id}_seg.nii.gz')).get_fdata()

#         # Convert to torch tensor
#         # todo: image, label의 가운데 slice를 가져와서 학습 데이터로 만들기

#         middle_slice_index = image.shape[2] // 2
#         image_slice = image[:, :, middle_slice_index]
#         label_slice = label[:, :, middle_slice_index]

#         # Convert to torch tensor
#         image_tensor = torch.tensor(image_slice, dtype=torch.float32).unsqueeze(0)  # Add channel dimension
#         label_tensor = torch.tensor(label_slice, dtype=torch.long)

#         return image_tensor, label_tensor

#     def __len__(self):
#         return len(self.patient_ids)

In [ ]:
data_dir = '../data/BraTS_2018_Train'  # put the directory of data
modalities = ['t1', 't1ce', 't2', 'flair']

In [ ]:
t1_dataset = Custom2DBraTSDataset(data_dir=data_dir, modality='t1')
len(t1_dataset)

In [ ]:
# datasets = {modality: Custom2DBraTSDataset(data_dir=data_dir, modality=modality) for modality in modalities}

In [ ]:
t1_dataloader = DataLoader(t1_dataset, batch_size=4, shuffle=True)
# t2_dataloader = DataLoader(datasets['t2'], batch_size=2, shuffle=True)
# t1ce_dataloader = DataLoader(datasets['t1ce'], batch_size=2, shuffle=True)
# flair_dataloader = DataLoader(datasets['flair'], batch_size=2, shuffle=True)

In [ ]:
for images, labels in t1_dataloader:
    print(images.shape, images)  # 출력: [batch_size, 1, D, H, W]
    print(labels.shape, labels)  # 출력: [batch_size, D, H, W]
    break

In [ ]:
# print(len(t1_dataloader), len(datasets['t1']))

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class RES_Block(nn.Module):

    def __init__(self, in_channels, out_channels):
        super(RES_Block, self).__init__()

        self.split_conv_x1_1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=(15, 1), padding=(7, 0)),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
            )
        self.split_conv_x1_2 = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, kernel_size=(1, 15), padding=(0, 7)),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
            )

        self.split_conv_x2_1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=(13, 1), padding=(6, 0)),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
            )
        self.split_conv_x2_2 = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, kernel_size=(1, 13), padding=(0, 6)),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
            )

        self.split_conv_x3_1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=(11, 1), padding=(5, 0)),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
            )
        self.split_conv_x3_2 = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, kernel_size=(1, 11),padding=(0, 5)),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
            )

        self.split_conv_x4_1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=(9, 1), padding=(4, 0)),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
            )
        self.split_conv_x4_2 = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, kernel_size=(1, 9), padding=(0, 4)),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
            )

        self.sum_conv_x1 = nn.Sequential(
            nn.Conv2d(5 * out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
            )
        self.sum_conv_x2 = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
            )
        self.sum_conv_x3 = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, kernel_size=1, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
            )

    def forward(self, x):
        init = x

        split_conv_x1 = self.split_conv_x1_1(x)
        split_conv_x1 = self.split_conv_x1_2(split_conv_x1)

        split_conv_x2 = self.split_conv_x2_1(x)
        split_conv_x2 = self.split_conv_x2_2(split_conv_x2)

        split_conv_x3 = self.split_conv_x3_1(x)
        split_conv_x3 = self.split_conv_x3_2(split_conv_x3)

        split_conv_x4 = self.split_conv_x4_1(x)
        split_conv_x4 = self.split_conv_x4_2(split_conv_x4)


        x = torch.cat([init, split_conv_x1, split_conv_x2, split_conv_x3, split_conv_x4],dim=1)

        x = self.sum_conv_x1(x)
        x = self.sum_conv_x2(x)
        x = self.sum_conv_x3(x)

        return x


class WC_Block(nn.Module):

    def __init__(self, in_channels, out_channels):
        super(WC_Block, self).__init__()

        self.split_conv_x1_1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=(15, 1)),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
            )
        self.split_conv_x1_2 = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, kernel_size=(1, 15)),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
            )

        self.split_conv_x2_1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=(1, 15)),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
            )
        self.split_conv_x2_2 = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, kernel_size=(15, 1)),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
            )

        self.conv_sum = nn.Conv2d(2* out_channels, out_channels, 3, padding=1)
        self.batch_norm = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
    def forward(self, x):

        split_conv_x1 = self.split_conv_x1_1(x)
        split_conv_x1 = self.split_conv_x1_2(split_conv_x1)

        split_conv_x2 = self.split_conv_x2_1(x)
        split_conv_x2 = self.split_conv_x2_2(split_conv_x2)
        x = torch.cat([split_conv_x1, split_conv_x2],dim=1)

        x = self.conv_sum(x)
        x = self.batch_norm(x)
        x = self.relu(x)

        return x


def conv(in_channels, out_channels):

    conv_block = nn.Sequential(
        nn.Conv2d(in_channels, out_channels, 3, padding=1),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_channels, out_channels, 3, padding=1),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True)
    )

    return conv_block


class BU_net(nn.Module):
    def __init__(self, n_classes):
        super(BU_net, self).__init__()

        self.convDown1 = conv(3, 64)
        self.convDown2 = conv(64, 128)
        self.convDown3 = conv(128, 256)
        self.convDown4 = conv(256, 512)
        self.convDown5 = nn.Sequential(
        nn.Conv2d(1024, 1024, 3, padding=1),
        nn.BatchNorm2d(1024),
        nn.ReLU(inplace=True)
        )
        self.maxpool = nn.MaxPool2d(2, stride=2)
        self.convUp4 = conv(1024+512, 512)
        self.convUp3 = conv(512+256, 256)
        self.convUp2 = conv(256+128, 128)
        self.convUp1 = conv(128+64, 64)
        self.convUp_fin = nn.Conv2d(64, n_classes, kernel_size=1)

        self.upsample1 = nn.ConvTranspose2d(1024, 1024, kernel_size=32, stride=1)
        self.upsample2 = nn.ConvTranspose2d(512, 512, kernel_size=31, stride=1)
        self.upsample3 = nn.ConvTranspose2d(256, 256, kernel_size=61, stride=1)
        self.upsample4 = nn.ConvTranspose2d(128, 128, kernel_size=121, stride=1)

        self.RES1 = RES_Block(64, 64)
        self.RES2 = RES_Block(128, 128)
        self.RES3 = RES_Block(256, 256)
        self.RES4 = RES_Block(512, 512)
        self.WC = WC_Block(512, 1024)

        self.sigmoid_layer = nn.Sigmoid()

    def forward(self, x):
        conv1 = self.convDown1(x)
        x = self.maxpool(conv1)
        conv2 = self.convDown2(x)
        x = self.maxpool(conv2)
        conv3 = self.convDown3(x)
        x = self.maxpool(conv3)
        conv4 = self.convDown4(x)
        x = self.maxpool(conv4)
        WC_5 = self.WC(x)
        conv5 = self.convDown5(WC_5)
        x = self.upsample1(conv5)

        RES_4 = self.RES4(conv4)
        x = torch.cat([RES_4,x], dim=1)
        x = self.convUp4(x)
        x = self.upsample2(x)

        RES_3 = self.RES3(conv3)
        x = torch.cat([RES_3,x], dim=1)
        x = self.convUp3(x)
        x = self.upsample3(x)

        RES_2 = self.RES2(conv2)
        x = torch.cat([RES_2,x], dim=1)
        x = self.convUp2(x)
        x = self.upsample4(x)

        RES_1 = self.RES1(conv1)
        x = torch.cat([RES_1,x], dim=1)
        x = self.convUp1(x)
        out = self.convUp_fin(x)

        out = self.sigmoid_layer(out)

        return out

In [ ]:
import torch
from torch.nn import functional as F
from torch.autograd import Function


def Dice_Loss_Coefficient(pred, target, weights, smooth=1.):
    pred = pred.contiguous()
    target = target.contiguous()
    weights = weights.contiguous()
    intersection = (pred * target * weights).sum(dim=2).sum(dim=2)
    union = (weights * pred).sum(dim=2).sum(dim=2) + (weights * target).sum(dim=2).sum(dim=2)

    loss = (1 - ((2. * intersection + smooth) / (union + smooth)))

    return loss.mean()


class Weighted_Cross_Entropy_Loss(torch.nn.Module):

    def __init__(self):
        super(Weighted_Cross_Entropy_Loss, self).__init__()

    def forward(self, pred, target, weights):
        n, c, H, W = pred.shape
        # Calculate log probabilities
        logp = F.log_softmax(pred, dim=1)

        # Gather log probabilities with respect to target
        logp = torch.gather(logp, 1, target.view(n, 1, H, W))

        # Multiply with weights
        weighted_logp = (logp * weights).view(n, -1)

        # Rescale so that loss is in approx. same interval
        weighted_loss = weighted_logp.sum(1) / weights.view(n, -1).sum(1)

        # Average over mini-batch
        weighted_loss = -weighted_loss.mean()

        return weighted_loss

class BU_Net_Loss(torch.nn.Module):
    def __init__(self, weight=None):
        super(BU_Net_Loss, self).__init__()
        self.weight = weight
        self.cross_entropy_loss = Weighted_Cross_Entropy_Loss(weight)

    def forward(self, pred, target):
        weights = self.compute_class_weight(target)
        wce_loss = self.cross_entropy_loss(pred, target, weights)
        dice_loss = Dice_Loss_Coefficient(pred, target, weights)
        total_loss = wce_loss + dice_loss
        return total_loss

    def compute_class_weight(self, target):
        n, H, W = target.size()
        class_weights = torch.zeros(n, H, W).to(target.device)
        for i in range(target.max() + 1):
            mask = (target == i).float()
            class_weight = 1.0 / (mask.sum() + 1e-6)
            class_weights += mask * class_weight
        return class_weights

In [ ]:
from tqdm import tqdm

In [ ]:
def train_model(trainloader, model, criterion, optimizer, device):
    model.train()
    for i, (inputs, labels) in tqdm(enumerate(trainloader), total = len(trainloader)):
        from datetime import datetime

        inputs = inputs.to(device)
        labels = labels.to(device=device, dtype=torch.int64)
        ################################################################################
        # TODO:                                                                        #
        # Fill In the code                                                             #
        ################################################################################
        inputs = inputs.repeat(1, 3, 1, 1)
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        ################################################################################
        #                                 END OF YOUR CODE                             #
        ################################################################################

In [ ]:

def accuracy_check(label, pred):
    ims = [label, pred]
    np_ims = []
    for item in ims:
        item = np.array(item)
        np_ims.append(item)
    compare = np.equal(np_ims[0], np_ims[1])
    accuracy = np.sum(compare)
    return accuracy / len(np_ims[0].flatten())

def accuracy_check_for_batch(labels, preds, batch_size):
    total_acc = 0
    for i in range(batch_size):
        total_acc += accuracy_check(labels[i], preds[i])
    return total_acc/batch_size

In [ ]:
def get_loss_train(model, trainloader, criterion, device):

    model.eval()
    total_acc = 0
    total_loss = 0
    for batch, (inputs, labels) in tqdm(enumerate(trainloader), total = len(trainloader)):
        with torch.no_grad():
            inputs = inputs.to(device)
            labels = labels.to(device = device, dtype = torch.int64)
            inputs = inputs.float()
            ################################################################################
            # TODO:                                                                        #
            # Fill In the code                                                             #
            ################################################################################

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            ################################################################################
            #                                 END OF YOUR CODE                             #
            ################################################################################
            outputs = np.transpose(outputs.cpu(), (0,2,3,1))
            preds = torch.argmax(outputs, dim=3).float()
            acc = accuracy_check_for_batch(labels.cpu(), preds.cpu(), inputs.size()[0])
            total_acc += acc
            total_loss += loss.cpu().item()
    return total_acc/(batch+1), total_loss/(batch+1)

In [ ]:
def val_model(model, valloader, criterion, device):

    total_val_loss = 0
    total_val_acc = 0
    n=0

    for batch, (inputs, labels) in tqdm(enumerate(valloader), total = len(valloader)):
        with torch.no_grad():

            inputs = inputs.to(device)
            labels = labels.to(device=device, dtype=torch.int64)
            ################################################################################
            # TODO:                                                                        #
            # Fill In the code                                                             #
            ################################################################################

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            ################################################################################
            #                                 END OF YOUR CODE                             #
            ################################################################################

            outputs = np.transpose(outputs.cpu(), (0, 2, 3, 1))
            preds = torch.argmax(outputs, dim=3).float()

            acc = accuracy_check_for_batch(labels.cpu(), preds.cpu(), inputs.size()[0])
            total_val_acc += acc
            total_val_loss += loss.cpu().item()



    return total_val_acc/(batch+1), total_val_loss/(batch+1)

In [ ]:
# Hyperparameter
################################################################################
# TODO:                                                                        #
# Hyperparameter Tuning                                                        #
################################################################################

batch_size = 16
learning_rate = 0.01
momentum = 0.9
epochs = 1

################################################################################
#                                 END OF YOUR CODE                             #
################################################################################

# 모델 초기화
model = BU_net(4)
print(model)

criterion = BU_Net_Loss
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)
model = model.to(device)

history = {'train_loss':[], 'train_acc':[]}

In [ ]:
print("Training")
for epoch in range(epochs):

    train_model(t1_dataloader, model, criterion, optimizer, device)
    train_acc, train_loss = get_loss_train(model, t1_dataloader, criterion, device)
    print("epoch", epoch + 1, "train loss : ", train_loss, "train acc : ", train_acc)

    # val_acc, val_loss = val_model(model, validLoader, criterion, device)
    # print("epoch", epoch + 1, "val loss : ", val_loss, "val acc : ", val_acc)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    # history['val_loss'].append(val_loss)
    # history['val_acc'].append(val_acc)

    if epoch % 5 == 0:
        torch.save(model.state_dict(), f'./{str(epoch)}.pth')

print('Finish Training')